In [11]:
from unicodedata import decomposition

from langchain_classic.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_classic.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain.chat_models import init_chat_model
from langchain_classic.prompts import PromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains.retrieval import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableSequence

from Conversational_Memory.conversational_memory import qa_prompt

In [12]:
loader = TextLoader("Langchain_crewai.txt")
docs = loader.load()

In [13]:
splitter = RecursiveCharacterTextSplitter(chunk_size= 300, chunk_overlap=50)
chunks = splitter.split_documents(docs)

embedding = HuggingFaceEmbeddings(model_name = "all-MiniLM-L6-v2")
vectorstore = FAISS.from_documents(chunks, embedding)

retriever = vectorstore.as_retriever(search_type = "mmr", search_kwargs = {"k":5, "lambda_mult":0.7})

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7238.22it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [14]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")


In [15]:
llm = init_chat_model('groq:openai/gpt-oss-120b', temperature=0.7)
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x11e0d7950>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x11e3e9390>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [16]:
decomposition_prompt = PromptTemplate.from_template(
    """You are a helpful assistant. Your task is to decompose the user's query into smaller sub-queries that can help retrieve more relevant documents.

    Original Query: "{question}"

    Sub-Queries: """
)

In [17]:
decomposition_chain = decomposition_prompt|llm|StrOutputParser()

In [18]:
query = "How does Langchain use memory and agents compared to CrewAI"

decomposition_question = decomposition_chain.invoke({"question": query})

In [20]:
print(decomposition_question)

**Sub‑Queries**

1. **LangChain Memory**
   - What types of memory modules does LangChain provide (e.g., conversation, vector, summary, token‑window, etc.)?
   - How are memory objects instantiated and attached to a LangChain chain or agent?
   - What are the default persistence options for LangChain memory (in‑memory, Redis, SQLite, etc.)?
   - How does LangChain handle memory cleanup, expiration, or context length limits?

2. **LangChain Agents**
   - What are the main agent classes in LangChain (e.g., `ZeroShotAgent`, `ReactAgent`, `ConversationalAgent`, `ToolCallingAgent`)?
   - How do agents decide which tools or LLM calls to invoke?
   - How is the agent’s reasoning loop (plan → act → observe → revise) implemented in LangChain?
   - Which built‑in tools and integrations does LangChain support for agents (search, APIs, code execution, etc.)?

3. **CrewAI Memory**
   - What memory mechanisms does CrewAI offer for its autonomous agents (e.g., shared state, task‑specific context, lon

In [21]:
qa_prompt = PromptTemplate.from_template(
    """
    Use the context below to answer the question. If you don't know the answer, say you don't know.

    Context:{context}

    question: {input}"""
)

qa_chain = create_stuff_documents_chain(llm = llm, prompt = qa_prompt)

In [ ]:
def full_query_rag_pipeline(user_query):
    sub_qs = decomposition_chain.invoke({"question": user_query})
    sub_questions = [q.strip("-1234567890")]
